# 📊 Daily Odin Global User Report Generator (Colab Edition) — **v1**

This interactive notebook wraps **`daily_odin_report.py`** and breaks it into easy‑to‑test units so you can:

* configure variables in one place  
* run each functional step independently  
* experiment with different report windows, batch sizes, etc.  

> **Important**: **Upload and import the script first** (see next section), then edit the configuration.


## 🗃️ Process & Aggregate User Report Data

In [1]:
# @@ Install / upgrade required libraries (Colab already has pandas/requests) @@
!pip -q install paramiko python-dotenv tqdm pandas

## Configuration

Edit the cell below to set **environment variables** or direct constants that the script relies on.


In [ ]:
# @title Configuration {"run":"auto"}

# @@ Runtime settings (edit freely) @@
import os, json
from dataclasses import asdict

# --- Minimum required ---
ODIN_API_BASE_URL = "" # @param {"type":"string"}
ODIN_API_USERNAME = "" # @param {"type":"string"}
ODIN_API_PASSWORD = "" # @param {"type":"string"}
start_date = "2025-07-01" # @param {"type":"date"}
end_date = "2025-07-01" # @param {"type":"date"}
sFTP_HOST = "" # @param {"type":"string"}
sFTP_USERNAME = "" # @param {"type":"string"}
sFTP_PASSWORD = "" # @param {"type":"string"}
SMTP_HOST = "" # @param {"type":"string"}
SMTP_USERNAME = "" # @param {"type":"string"}
SMTP_PASSWORD = "" # @param {"type":"string"}
SMTP_FROM = "" # @param {"type":"string","placeholder":"report@example.com"}
SMTP_TO = "" # @param {"type":"string","placeholder":"you@example.com"}

start_date = start_date + " 00:00:00"
end_date = end_date + " 23:59:59"

# --- Minimum required ---
os.environ.setdefault('ODIN_API_BASE_URL', ODIN_API_BASE_URL)
os.environ.setdefault('ODIN_API_USERNAME', ODIN_API_USERNAME)
os.environ.setdefault('ODIN_API_PASSWORD', ODIN_API_PASSWORD)

# --- Optional overrides ---
os.environ.setdefault('REPORT_START_DATE', start_date) # e.g. '2025-06-01 00:00:00'
os.environ.setdefault('REPORT_END_DATE', end_date) # e.g. '2025-06-01 23:59:59'
os.environ.setdefault('BATCH_SIZE', '200')

# SFTP (leave blank to disable)
os.environ.setdefault('SFTP_HOST', sFTP_HOST)
os.environ.setdefault('SFTP_USERNAME', sFTP_USERNAME)
os.environ.setdefault('SFTP_PASSWORD', sFTP_PASSWORD)

# SMTP (leave blank to disable)
os.environ.setdefault('SMTP_HOST', SMTP_HOST)
os.environ.setdefault('SMTP_USERNAME', SMTP_USERNAME)
os.environ.setdefault('SMTP_PASSWORD', SMTP_PASSWORD)
os.environ.setdefault('SMTP_FROM', SMTP_FROM)
os.environ.setdefault('SMTP_TO', SMTP_TO)

## 📥 Load `daily_odin_report.py`

* If running in Colab, upload the script when prompted below (first‑time run)  
* If the script lives on GitHub/GCS, you can `wget`/`gsutil cp` instead.


In [ ]:
# === Fresh-upload & import daily_odin_report.py ============================
import os, shutil, importlib.util, sys, re, textwrap
from google.colab import files

# 1️⃣  Make sure any old copy is removed so we *must* upload a new one
for fname in ("daily_odin_report.py",):
    if os.path.exists(fname):
        os.remove(fname)

# 2️⃣  Prompt user to upload a fresh file every run
uploaded = files.upload()                      # ⇧ choose your latest daily_odin_report.py
if not uploaded:
    raise FileNotFoundError("You must upload daily_odin_report.py to continue")

src_name = next(iter(uploaded))                # first (and usually only) uploaded file
shutil.move(src_name, "daily_odin_report.py")  # rename / overwrite

# 3️⃣  Hot-patch the mutable-default list issue, if still present
path = "daily_odin_report.py"
txt  = open(path).read()

if "smtp_to: List[str] =" in txt and "field(" not in txt:       # unpatched version
    # ensure 'field' is imported
    if "from dataclasses import" in txt:
        txt = re.sub(r"from dataclasses import ([^\n]+)",
                     lambda m: ("field" in m.group(1) and m.group(0) or
                                 f"from dataclasses import {m.group(1)}, field"),
                     txt, 1)
    else:
        txt = "from dataclasses import field\n" + txt

    # replace smtp_to line with default_factory version
    txt = re.sub(
        r"smtp_to\s*:\s*List\[str\]\s*=\s*[^\n]+",
        textwrap.dedent("""\
            smtp_to: List[str] = field(
                default_factory=lambda: [
                    a.strip() for a in os.getenv('SMTP_TO', '').split(',') if a.strip()
                ]
            )"""),
        txt, 1
    )
    open(path, "w").write(txt)
    print("🔧  Applied mutable-default patch")

# 4️⃣  Dynamic import
spec = importlib.util.spec_from_file_location("daily_odin_report", path)
dcr  = importlib.util.module_from_spec(spec)
sys.modules["daily_odin_report"] = dcr
spec.loader.exec_module(dcr)

print("✅  daily_odin_report.py uploaded & imported fresh")

## 📥 Load `daily_odin_report.py` local

In [ ]:
# === Load daily_odin_report.py from local filesystem (no upload required) ===
import os, importlib.util, sys

path = "daily_odin_report.py"
assert os.path.exists(path), f"{path} not found in current directory!"

# Dynamic import
spec = importlib.util.spec_from_file_location("daily_odin_report", path)
dcr  = importlib.util.module_from_spec(spec)
sys.modules["daily_odin_report"] = dcr
spec.loader.exec_module(dcr)

print("✅ daily_odin_report.py loaded from local filesystem and imported as 'dcr'")

## 🔑 Authenticate & Create API Client

In [ ]:
cfg = dcr.Config()
print(json.dumps(asdict(cfg), indent=2))
cfg = dcr.Config()                       # pick up env vars
api_client = dcr.OdinAPIClient(cfg)
assert api_client.authenticate(), "API authentication failed ❌"
print("Authenticated ✔")

## 🏢 Fetch All Service Providers

In [ ]:
# Get all Service Providers
service_providers = api_client.get_service_providers()
len(service_providers), service_providers[:5]

## 🏢 Fetch Service Providers from list

In [ ]:
# Get Service Providers from the list
SERVICE_PROVIDER_LIST = ["ent.odin","ent.odin.demo"] # @param {"type":"string"}

all_sps = api_client.get_service_providers()
service_providers = [
    sp for sp in all_sps
    if sp.get('serviceProviderId') in SERVICE_PROVIDER_LIST
]
len(service_providers), service_providers[0] if service_providers else None

## 🌍 Fetch Global User Report for Service Providers

In [ ]:
from tqdm.auto import tqdm
import pandas as pd

# Fetch comprehensive global user data from ALL service providers
print("Fetching global user report from all service providers...")
global_users = api_client.get_global_user_report()

print(f"\nTotal users in global report: {len(global_users):,}")

# Display sample global user data
if global_users:
    print("\nSample global user data:")
    sample_user = global_users[0]
    for key, value in list(sample_user.items())[:10]:  # Show first 10 fields
        print(f"  {key}: {value}")
    
    # Convert to DataFrame for easier viewing
    df_global_users = pd.DataFrame(global_users)
    print(f"\nDataFrame shape: {df_global_users.shape}")
    print("\nColumns:", list(df_global_users.columns))
    
    # Show summary by service provider
    if 'serviceProviderId' in df_global_users.columns:
        sp_summary = df_global_users.groupby('serviceProviderId').size()
        print("\nUsers per Service Provider (Global Report):")
        for sp_id, count in sp_summary.items():
            print(f"  {sp_id}: {count} users")
        
        # Show total unique service providers
        unique_sps = df_global_users['serviceProviderId'].nunique()
        print(f"\nTotal unique Service Providers: {unique_sps}")
        
        # Show unique groups if available
        if 'groupId' in df_global_users.columns:
            unique_groups = df_global_users['groupId'].nunique()
            print(f"Total unique Groups: {unique_groups}")
    
    # Optional: Process with GlobalUserDataProcessor
    PROCESS_GLOBAL_DATA = True # @param {"type":"boolean"}
    
    if PROCESS_GLOBAL_DATA:
        print("\nProcessing global user data...")
        processor = dcr.GlobalUserDataProcessor(include_optional_fields=True)
        df_clean, df_summary = processor.process_user_data(global_users)
        
        print(f"Processed data shape: {df_clean.shape}")
        print(f"Summary data shape: {df_summary.shape}")
        
        if not df_summary.empty:
            print("\nSummary by Service Provider:")
            print(df_summary.head())
else:
    print("No global user data found")

## 🗃️ Process & Aggregate Global User Report

In [ ]:
## 🌍 Process & Aggregate Global User Report

# If you have run the "Fetch Global User Report for Service Providers" cell and have df_global_users:
if 'df_global_users' in globals():
    print("Processing and aggregating global user report data...")
    processor = dcr.GlobalUserDataProcessor(include_optional_fields=True)
    df_global_clean, df_global_summary = processor.process_user_data(df_global_users.to_dict(orient='records'))
    print(f"Processed global user data shape: {df_global_clean.shape}")
    print(f"Global user summary data shape: {df_global_summary.shape}")
    if not df_global_summary.empty:
        print("\nGlobal User Summary by Service Provider:")
        print(df_global_summary.head())
else:
    print("No global user report data found. Please run the global user report fetch cell first.")

## 💾 Export Global User Report to CSV

In [ ]:
## 💾 Export Global User Report to CSV

# If you have run the "Process & Aggregate Global User Report" cell and have df_global_clean and df_global_summary:
if 'df_global_clean' in globals() and 'df_global_summary' in globals():
    exporter = dcr.ReportExporter(cfg)
    global_raw_csv, global_summary_csv = exporter.export_user_data_to_csv(df_global_clean, df_global_summary)
    print('Global User Report Files:', global_raw_csv, global_summary_csv)
else:
    print("No processed global user report data found. Please run the global user report processing cell first.")

## ☁️ Optional Outputs (SFTP Upload & Email)

In [ ]:
uploaded = False; emailed = False
if cfg.sftp_host:
    uploaded = exporter.upload_to_sftp([raw_csv, agg_csv])
if cfg.smtp_host:
    summary = dcr.DailyCallReportGenerator(cfg)._calculate_summary_stats(
        df_raw, df_agg, len(service_providers), total_users
    )
    emailed = exporter.send_email_report([raw_csv, agg_csv], summary)
print('SFTP:', uploaded, 'Email:', emailed)

---
### ✅ Notebook Complete
Generated 2025-06-27 21:03 UTC